## Cargar los datos ##

In [257]:
import numpy as np
import pandas as pd

In [258]:
df = pd.read_csv('data/GenreAll.csv', skipinitialspace=True)

# eliminar filas que no tienen género
df = df.dropna(subset=['genre']) 

#feature que queremos predecir
y = df['genre'] 

#features para el entrenamiento
X = df.drop(columns=["genre"]).copy()

# Convertir columnas binarias de texto en 0 y 1
text_columns = X.select_dtypes(include=['str', 'object', 'string']).columns

for col in text_columns:
    values = X[col].dropna().unique()

    if len(values) == 2:
        X[col] = X[col].map({
            values[0]: 0,
            values[1]: 1})
            
#asegurar que solo hay valores numéricos
X = X.select_dtypes(include=np.number)

print("Número de canciones:", len(df))
print("Número de features:", X.shape[1])
print("\nCanciones por género:")
print(y.value_counts())

df.head()

Número de canciones: 993
Número de features: 240

Canciones por género:
genre
blu    100
cou    100
dis    100
hip    100
jaz    100
met    100
pop    100
reg    100
roc    100
cla     93
Name: count, dtype: int64


,highlevel_danceability_value,highlevel_equalization_profile_value,highlevel_excitement_value,highlevel_intensity_value,highlevel_speech_music_value,highlevel_voice_instrumental_value,loudness_dynamic_complexity_dvar,loudness_dynamic_complexity_mean,loudness_dynamic_complexity_var,loudness_larm_dvar,...,tonal_chords_strength_dvar,tonal_chords_strength_mean,tonal_chords_strength_var,tonal_dissonance_dvar,tonal_dissonance_mean,tonal_dissonance_var,tonal_key_mode_value,tonal_key_strength_value,tonal_tuning_equal_tempered_deviation_value,genre
0,0.477744,0.372766,0.029087,0.093526,speech,instrumental,-0.366526,0.832351,-0.398487,0.262413,...,0.175405,0.101673,0.989475,0.167484,-0.091051,0.085449,major,0.162259,2.423830,blu
1,0.271161,0.372766,-1.283803,-0.615420,speech,voice,0.275184,1.901008,-0.287263,0.078360,...,-0.157583,-0.652126,-0.992276,0.606661,-0.468630,0.416914,major,0.230571,-0.207952,blu
2,0.089011,0.372766,0.029087,1.511419,music,instrumental,-0.177390,0.046932,0.035041,-0.157974,...,-0.001059,-0.220255,0.978014,-0.236338,0.470580,-0.205887,major,1.024735,-0.211034,blu
3,0.243503,0.372766,0.029087,-0.615420,music,instrumental,-0.319420,-0.953075,0.239607,-0.659566,...,0.214463,0.853728,1.327351,-0.542341,0.642223,-0.791110,major,0.079033,0.148357,blu
4,0.594883,-1.940715,-1.283803,0.093526,speech,instrumental,-0.224956,1.154507,-0.420087,0.281666,...,-0.140930,-0.080519,-1.163700,0.094741,-0.164770,-0.277919,major,-0.337966,-0.277459,blu


## Definición de los modelos ##

Un pipeline es una cadena de pasos que se ejecutan en orden. Por ejemplo, antes de entrenar un modelo, podemos necesitar rellenar datos que faltan y normalizar las características.

`Pipeline` permite crear un pipeline combinando varios pasos

`SimpleImputer` sirve para rellenar los valores que faltan en los datos, por ejemplo, cuando una característica musical contiene NaN. se puede usar `StandardScaler` para rellenar esos datos vacíos

In [259]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

#modelos
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier

In [260]:
models = {
    "Logistic Regression": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000))
        ]),
        {
            "model__C": [0.01, 0.03, 0.1, 0.3, 1]
        }
    ),

    "k-NN": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", KNeighborsClassifier())
        ]),
        {
            "model__n_neighbors": [3, 5, 7, 9, 13, 17],
            "model__weights": ["uniform", "distance"],
            "model__p": [1, 2]
        }
    ),

    "Decision Tree": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", DecisionTreeClassifier(random_state=42))
        ]),
        {
            "model__max_depth": [5, 10, 15],
            "model__min_samples_split": [5, 10],
            "model__min_samples_leaf": [5, 10]
        }
    ),

    "Artificial Neural Network": (
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                max_iter=500,
                random_state=42,
                early_stopping=False,
                n_iter_no_change=15,
                tol=1e-3
            ))
        ]),
        {
            "model__hidden_layer_sizes": [
                (30,), (50,), (100,), (50, 25)
            ],
            "model__alpha": [0.01, 0.1, 1.0]
        }
    )
}

## Definir la validación cruzada
 
Mismo esquema de 10 folds para comparar todos los algoritmos

In [261]:
from sklearn.model_selection import StratifiedKFold

In [262]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42)

## Entrenar y evaluar los modelos

Esta es la celda principal. `GridSearchCV` probará todas las configuraciones usando validación cruzada de 10 folds y seleccionará la que obtenga la mejor accuracy media para cada algoritmo

In [263]:
from sklearn.model_selection import GridSearchCV

In [264]:

results = []
best_models = {}

for name, (pipeline, params) in models.items():

    print("\nEvaluando:", name)

    search = GridSearchCV(
        estimator=pipeline,
        param_grid=params,
        scoring="accuracy",
        cv=cv,
        return_train_score=True,
        n_jobs=-1)

    search.fit(X, y)

    best_models[name] = search

    # Guardar los resultados de todas las configuraciones
    for i in range(len(search.cv_results_["params"])):
        results.append({
            "Algorithm": name,
            "Parameters": search.cv_results_["params"][i],
            "Train accuracy": search.cv_results_["mean_train_score"][i],
            "CV accuracy": search.cv_results_["mean_test_score"][i],
            "CV standard deviation": search.cv_results_["std_test_score"][i]})

    print("Mejores parámetros:", search.best_params_)
    print("Best 10-fold CV accuracy:", round(search.best_score_, 4))


results_df = pd.DataFrame(results)

results_df["Overfitting gap"] = (
    results_df["Train accuracy"] -
    results_df["CV accuracy"]
)

results_df["Fits objectives"] = (
    (results_df["CV accuracy"] > 0.70) &
    (results_df["Overfitting gap"] < 0.15)
)

results_df = results_df.sort_values(
    ["Fits objectives", "Overfitting gap", "CV accuracy"],
    ascending=[False, True, False]
).reset_index(drop=True)


print(f"Total de configuraciones evaluadas: {len(results_df)}")
print(f"Configuraciones con accuracy >= 80%: "
      f"{(results_df['CV accuracy'] >= 0.80).sum()}")
print(f"Configuraciones con accuracy >= 70% y gap <= 0.10: "
      f"{((results_df['CV accuracy'] >= 0.70) & (results_df['Overfitting gap'] <= 0.10)).sum()}")

display(results_df)



Evaluando: Logistic Regression
Mejores parámetros: {'model__C': 0.1}
Best 10-fold CV accuracy: 0.8408

Evaluando: k-NN
Mejores parámetros: {'model__n_neighbors': 5, 'model__p': 1, 'model__weights': 'distance'}
Best 10-fold CV accuracy: 0.7562

Evaluando: Decision Tree
Mejores parámetros: {'model__max_depth': 15, 'model__min_samples_leaf': 5, 'model__min_samples_split': 5}
Best 10-fold CV accuracy: 0.6423

Evaluando: Artificial Neural Network
Mejores parámetros: {'model__alpha': 1.0, 'model__hidden_layer_sizes': (100,)}
Best 10-fold CV accuracy: 0.8499
Total de configuraciones evaluadas: 53
Configuraciones con accuracy >= 80%: 17
Configuraciones con accuracy >= 70% y gap <= 0.10: 10


,Algorithm,Parameters,Train accuracy,CV accuracy,CV standard deviation,Overfitting gap,Fits objectives
0,k-NN,"{'model__n_neighbors': 17, 'model__p': 1, 'mod...",0.752936,0.704838,0.051164,0.048097,True
1,k-NN,"{'model__n_neighbors': 13, 'model__p': 1, 'mod...",0.767035,0.708859,0.052277,0.058176,True
2,k-NN,"{'model__n_neighbors': 9, 'model__p': 1, 'mode...",0.790533,0.728949,0.046849,0.061583,True
3,k-NN,"{'model__n_neighbors': 13, 'model__p': 2, 'mod...",0.764461,0.701848,0.055465,0.062612,True
4,k-NN,"{'model__n_neighbors': 9, 'model__p': 2, 'mode...",0.781692,0.712869,0.053060,0.068823,True
5,k-NN,"{'model__n_neighbors': 7, 'model__p': 2, 'mode...",0.795568,0.715879,0.050482,0.079689,True
6,k-NN,"{'model__n_neighbors': 5, 'model__p': 2, 'mode...",0.814142,0.731030,0.038827,0.083112,True
7,Logistic Regression,{'model__C': 0.01},0.892245,0.805606,0.039233,0.086639,True
8,k-NN,"{'model__n_neighbors': 7, 'model__p': 1, 'mode...",0.810897,0.723919,0.055035,0.086978,True
9,k-NN,"{'model__n_neighbors': 5, 'model__p': 1, 'mode...",0.830143,0.739091,0.046500,0.091052,True
